### 지도 학습 지시 미세 튜닝을 위해 데이터셋 준비하기

In [2]:
import json

with open("instruction-data.json", "r") as file:
    data = json.load(file)
print(f"샘플 개수: {len(data)}")

샘플 개수: 1100


In [3]:
# 코드 7-2 프롬프트 포맷팅 함수 구현하기
def format_input(entry):
    instruction_text = (
        f"Below is an instruction that describes a task. "
        f"Write a response that appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}"
    )
    input_text = (
        f"\n\n### Input:\n{entry['input']}" if entry["input"] else ""
    )
    return instruction_text + input_text

In [4]:
# 코드 7-3 데이터셋 분할하기
train_portion = int(len(data) * 0.85)                   # 935
test_portion = int(len(data) * 0.1)                     # 110
val_portion = len(data) - train_portion - test_portion  # 55

train_data = data[:train_portion]                               # [:935]
test_data = data[train_portion:train_portion + test_portion]    # [935:1045]
val_data = data[train_portion+test_portion:]                    # [1045:]

print("훈련 세트 크기: ", len(train_data))
print("검증 세트 크기: ", len(val_data))
print("테스트 세트 크기: ", len(test_data))

훈련 세트 크기:  935
검증 세트 크기:  55
테스트 세트 크기:  110


### 훈련 배치 만들기

In [5]:
# 코드 7-4 지시 데이터셋 클래스 구현하기
import torch
from torch.utils.data import Dataset

class InstructionDataset(Dataset):
    def __init__(self, data, tokenizer):
        self.data = data
        self.encoded_texts = []
        for entry in data:
            instruction_plus_input = format_input(entry)
            response_text = f"\n\n### Response:\n{entry['output']}"
            full_text = instruction_plus_input + response_text
            self.encoded_texts.append(tokenizer.encode(full_text))

    def __getitem__(self, index):
        return self.encoded_texts[index]

    def __len__(self):
        return len(self.data)

In [6]:
# 코드 7-5 사용자 정의 콜레이트 함수 구현하기
def custom_collate_fn(batch, pad_token_id=50256, ignore_index=-100, allowed_max_length=None, device="cpu"):
    batch_max_length = max(len(item)+1 for item in batch)
    inputs_lst, targets_lst = [], []

    for item in batch:
        new_item = item.copy()
        new_item += [pad_token_id]
        padded = (new_item + [pad_token_id] * (batch_max_length - len(new_item)))
        inputs = torch.tensor(padded[:-1])
        targets = torch.tensor(padded[1:])

        mask = targets == pad_token_id
        indices = torch.nonzero(mask).squeeze()
        if indices.numel() > 1:
            targets[indices[1:]] = ignore_index

        if allowed_max_length is not None:
            inputs = inputs[:allowed_max_length]
            targets = targets[:allowed_max_length]

        inputs_lst.append(inputs)
        targets_lst.append(targets)

    inputs_tensor = torch.stack(inputs_lst).to(device)
    targets_tensor = torch.stack(targets_lst).to(device)
    return inputs_tensor, targets_tensor

### 지시 데이터셋을 위한 데이터 로더 만들기

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [10]:
from functools import partial

customized_collate_fn = partial(
    custom_collate_fn,
    device=device,
    allowed_max_length=1024
)

In [ ]:
# 코드 7-6 데이터 로더 초기화하기
import tiktoken
from torch.utils.data import DataLoader


num_workers = 0
batch_size = 8
tokenizer = tiktoken.get_encoding("gpt2")

torch.manual_seed(123)

train_dataset = InstructionDataset(train_data, tokenizer)
train_loader = DataLoader(
    # dataset: Dataset[Unknown],
    dataset=train_dataset,
    # batch_size: int | None = 1,
    batch_size=batch_size,
    # shuffle: bool | None = None,
    shuffle=True,
    # sampler: Sampler[Unknown] | Iterable[Unknown] | None = None,
    # batch_sampler: Sampler[list[Unknown]] | Iterable[list[Unknown]] | None = None,
    # num_workers: int = 0,
    num_workers=0,
    # collate_fn: _collate_fn_t[Unknown] | None = None,
    collate_fn=customized_collate_fn,
    # pin_memory: bool = False,
    # drop_last: bool = False,
    drop_last=True,
    # timeout: float = 0,
    # worker_init_fn: _worker_init_fn_t | None = None,
    # multiprocessing_context: Unknown | None = None,
    # generator: Unknown | None = None,
    # *,
    # prefetch_factor: int | None = None,
    # persistent_workers: bool = False,
    # pin_memory_device: str = "",
    # in_order: bool = True
)

val_dataset = InstructionDataset(val_data, tokenizer)
val_loader = DataLoader(
    dataset=val_dataset,
    batch_size=batch_size,
    collate_fn=customized_collate_fn,
    shuffle=False,
    drop_last=False,
    num_workers=num_workers,
)

test_dataset = InstructionDataset(test_data, tokenizer)
test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=batch_size,
    collate_fn=customized_collate_fn,
    shuffle=False,
    drop_last=False,
    num_workers=num_workers,
)

In [12]:
# 훈련 데이터 로더에서 생성된 입력 배치와 타깃 배치의 차원을 확인해 보자
print("훈련 데이터 로더:")
for inputs, targets in train_loader:
    print(inputs.shape, targets.shape)

훈련 데이터 로더:
torch.Size([8, 61]) torch.Size([8, 61])
torch.Size([8, 76]) torch.Size([8, 76])
torch.Size([8, 73]) torch.Size([8, 73])
torch.Size([8, 68]) torch.Size([8, 68])
torch.Size([8, 65]) torch.Size([8, 65])
torch.Size([8, 72]) torch.Size([8, 72])
torch.Size([8, 80]) torch.Size([8, 80])
torch.Size([8, 67]) torch.Size([8, 67])
torch.Size([8, 62]) torch.Size([8, 62])
torch.Size([8, 75]) torch.Size([8, 75])
torch.Size([8, 62]) torch.Size([8, 62])
torch.Size([8, 68]) torch.Size([8, 68])
torch.Size([8, 67]) torch.Size([8, 67])
torch.Size([8, 77]) torch.Size([8, 77])
torch.Size([8, 69]) torch.Size([8, 69])
torch.Size([8, 79]) torch.Size([8, 79])
torch.Size([8, 71]) torch.Size([8, 71])
torch.Size([8, 66]) torch.Size([8, 66])
torch.Size([8, 83]) torch.Size([8, 83])
torch.Size([8, 68]) torch.Size([8, 68])
torch.Size([8, 80]) torch.Size([8, 80])
torch.Size([8, 71]) torch.Size([8, 71])
torch.Size([8, 69]) torch.Size([8, 69])
torch.Size([8, 65]) torch.Size([8, 65])
torch.Size([8, 68]) torch.Siz